# Missing data

## Imports

In [2]:
import os
from collections import OrderedDict
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append("../Jeremy")
from visualization import plot_df_histogram
sys.path.append("../Adrian")

## Dates

In [3]:
from datetime import datetime, timedelta

day1=datetime.strptime("2018.01.15","%Y.%m.%d")
print(day1)
day2=day1+timedelta(days=18)
print(day2)

2018-01-15 00:00:00
2018-02-02 00:00:00


In [4]:
import json

root="../../../newyorker_caption_contest/"

file_path = root+"contests.json"
with open(file_path, "r") as file:
    contests = json.load(file)

# Convert the JSON data into a DataFrame for easier manipulation
contests_df = pd.DataFrame(contests)
#print(contests_df['contest_id'])


contest_510_publish=datetime.strptime('2016.02.22',"%Y.%m.%d")
contest_510_votes=datetime.strptime('2016.02.29',"%Y.%m.%d")

for i in range(len(contests_df)):
    contest_id=contests_df['contest_id'].iloc[i]
    print(contest_id)
    diff=int(contest_id)-510
    contest_day_publish=contest_510_publish+timedelta(days=diff*7)
    print(contest_day_publish)
    contests_df['metadata'].iloc[i]['contest_day_publish']=contest_day_publish ## bug prcq pas de dico/missing values
    contests_df['metadata'].iloc[i]['contest_day_votes']=contest_day_publish+timedelta(days=7)
    break


print(contests_df['metadata'].keys())
print(contests_df.iloc[0,:]['metadata']['contest_day_publish'])


510
2016-02-22 00:00:00
RangeIndex(start=0, stop=384, step=1)
2016-02-22 00:00:00


## Add descritpions

Once you've installed ollama https://ollama.ai/ , and installed it in your environment, `pip install ollama`, run on youre terminal ollama serve, then on a new shell, pull the ollama model, in this case `ollama pull llava:7b` (4,4 GB), Then you should be able to run the code.

In [22]:
from tqdm import tqdm
import ollama
import base64
import random
import json
    
file_path = root+"contests.json"
with open(file_path, "r") as file:
    contests = json.load(file)

# Convert the JSON data into a DataFrame for easier manipulation
contests_df = pd.DataFrame(contests)


def missing_meta(meta, what):
    """
    Check if a single metadata dictionary is missing a given key.
    """
    return (
        not isinstance(meta, dict)
        or what not in meta
        )
    

'''
def ids_missing_key(metadata_list, what):
    """
    what can be one of the following : 'num_captions', 'num_votes', 'image_locations', 
    'image_descriptions', 'image_uncanny_descriptions', 'entities', 'questions', AJOUTER LES DATES
    """
    return [i for i, meta in enumerate(metadata_list) if missing_meta(meta, what)]

#missing_df_index = indices_missing_key(contests_df['metadata'],'image_descriptions')
#print(missing_df_index)
'''
def ids_missing_key(df, what):
    """
    Returns the contest IDs (not indices) for which the metadata dictionary is missing the key `what`.

    Args:
        metadata_list (list): List of metadata dicts (contests_df['metadata'])
        what (str): Key to check for (e.g. 'num_captions', 'image_descriptions', etc.)
        contest_ids (list or Series): List of contest IDs aligned with metadata_list
    """
    return [
        int(df['contest_id'][i])
        for i, meta in enumerate(df['metadata'])
        if missing_meta(meta, what)
    ]

def examples_for_ai(df,number_of_ex=5):
    examples = []
    for _, complete in random.sample(list(df.iterrows()),number_of_ex):
        image_path=root+'images/'+str(complete['contest_id'])+'.jpg'
        #with open(image_path, "rb") as f:
                #image_b64 = base64.b64encode(f.read()).decode("utf-8")
        meta = complete['metadata']
        
        examples.append({
            'image': image_path,'image_loc': meta.get('image_locations'),
            'image_descri': meta.get('image_descriptions'), 'image_uncan_descri':meta.get('image_uncanny_descriptions'),
            'images_entities': meta.get('entities'),'images_questions': meta.get('questions')
        })
    return examples

def prompt_making(examples):
    prompt = "You are an assistant that generates detailed and structured metadata for images.\n\n"
        
    for example in examples:  # Fewer examples for smaller models
        prompt += f"Example:\n"
        prompt += f"image_path: {example['image']}\n"
        prompt += f"Location: {example['image_loc']}\n"
        prompt += f"Description: {example['image_descri']}\n"
        prompt += f"Uncanny description : {example['image_uncan_descri']}\n"
        prompt += f"Entities: {example['images_entities']}\n"
        prompt += f"Questions: {example['images_questions']}\n\n"
    
    return prompt

def add_image_descriptions(contests_df):  
    print(1)
    # Test if Ollama is running and pull a model
    try:
        ollama.list()
    except:
        print("Please install and start Ollama first: https://ollama.ai/")
        return contests_df
    missing_df_ids = ids_missing_key(contests_df,'image_descriptions')
    complete_df = contests_df[~contests_df['contest_id'].isin(missing_df_ids)] ## removes the missings dfs
    #print(missing_df_index)
    print(2)

    required_keys = [
        "image_locations", "image_descriptions", "image_uncanny_descriptions",
        "entities", "questions",
    ]
    results = []
    for ids in tqdm(missing_df_ids):
        
        meta = contests_df.loc[contests_df['contest_id'] == ids, 'metadata'].iloc[0]
        if not isinstance(meta, dict):
            meta = {}
        print(3)
        image_path=root+'images/'+str(ids)+'.jpg'
        #with open(image_path, "rb") as f:
                #image_b64 = base64.b64encode(f.read()).decode("utf-8")
        print(4)
        examples = examples_for_ai(complete_df)    
        print(5)
        prompt=prompt_making(examples)
        print(6)
        prompt += f"""
        Incomplete metadata: {meta}
        Fill in the missing keys: {', '.join(required_keys)} for the following image {image_path}
        Provide a JSON object with all fields completed.
        Return ONLY valid JSON, no other text.
        """
        try:
            print(7)
            response = ollama.generate(
                model='llava:7b',  # or 'mistral', 'codellama', etc.
                prompt=prompt,
                options={'temperature': 0.1}  # control randomness/creativity, low temp mostly consistent, slight creativity
            )
            print(8)
            description = response['response']
        
            results.append({
            'contest_id': ids,
            'description': description
            })
            print(9)
        except Exception as e:
            print(f"Error processing contest {ids}: {e}")
            continue
        print(ids)
        print(description)
        break

    return contests_df

new=add_image_descriptions(contests_df)

#print(new)



1
2


  0%|          | 0/144 [00:00<?, ?it/s]

3
4
5
6
7


  0%|          | 0/144 [09:09<?, ?it/s]

8
9
527
 {
"image\_path": "../../../newyorker\_caption\_contest/images/527.jpg",
"image\_locations": ["a laboratory"],
"image\_descriptions": ["A scientist is working on a machine in his lab. He is surrounded by various equipment and chemicals.", "The scientist is wearing a white coat and goggles, indicating that he is conducting an experiment."],
"image\_uncanny\_descriptions": ["It's unusual to see a scientist working alone in a lab without any assistants or other personnel present."],
"entities": ["https://en.wikipedia.org/wiki/Scientist", "https://en.wikipedia.org/wiki/Lab"],
"questions": ["What is the scientist trying to achieve with his experiment?", "Why is he working alone in the lab?"]
} 


In [ ]:
#missing_df_ids = ids_missing_key(contests_df,'image_descriptions')
#print(contests_df['contest_id'])

#print(missing_df_ids)
image_path=root+'images/'+str(510)+'.jpg'
with open(image_path, "rb") as f:
    image_bytes = f.read()  # This gives you the raw bytes
    
print(image_bytes)
'''
with open(image_path, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")
    
print(image_b64)
'''

b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01\x00H\x00H\x00\x00\xff\xe1\x00XExif\x00\x00MM\x00*\x00\x00\x00\x08\x00\x03\x01\x06\x00\x03\x00\x00\x00\x01\x00\x02\x00\x00\x01\x12\x00\x03\x00\x00\x00\x01\x00\x01\x00\x00\x87i\x00\x04\x00\x00\x00\x01\x00\x00\x002\x00\x00\x00\x00\x00\x02\xa0\x02\x00\x04\x00\x00\x00\x01\x00\x00\x04\n\xa0\x03\x00\x04\x00\x00\x00\x01\x00\x00\x03\n\x00\x00\x00\x00\xff\xed\x008Photoshop 3.0\x008BIM\x04\x04\x00\x00\x00\x00\x00\x008BIM\x04%\x00\x00\x00\x00\x00\x10\xd4\x1d\x8c\xd9\x8f\x00\xb2\x04\xe9\x80\t\x98\xec\xf8B~\xff\xe2\x0c\xd8ICC_PROFILE\x00\x01\x01\x00\x00\x0c\xc8appl\x02\x10\x00\x00mntrRGB XYZ \x07\xdf\x00\x06\x00\x05\x00\x02\x00.\x00\x1cacspAPPL\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xf6\xd6\x00\x01\x00\x00\x00\x00\xd3-appl\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x

'\nwith open(image_path, "rb") as f:\n    image_b64 = base64.b64encode(f.read()).decode("utf-8")\n\nprint(image_b64)\n'